# Geospatial Features Extraction for Nigerian Cities

**Notebook 01a: JRC Water Bodies + OSM Roads & Places**

This notebook extracts geospatial features for city AOIs using:
- **JRC Global Surface Water** (via Google Earth Engine) — Water bodies with ≥50% occurrence
- **OpenStreetMap** (via Overpass API) — Roads and place names

---

## Table of Contents

1. [Load City AOIs](#1-load-city-aois) — Load AOI boundaries and check tracking status
2. [Extract Features for Each City](#2-extract-features-for-each-city) — Download water (JRC), roads (OSM), and places (OSM)
3. [Visualize Geospatial Features](#3-visualize-geospatial-features) — Create individual city maps
4. [Publication-Ready Study Area Map](#4-publication-ready-study-area-map) — Multi-panel figure for manuscripts
5. [Summary](#summary) — Workflow recap and next steps

---

## Data Sources

| Feature | Source | Resolution | Method |
|---------|--------|------------|--------|
| **Water Bodies** | JRC Global Surface Water | 30m | Google Earth Engine (≥50% occurrence) |
| **Roads** | OpenStreetMap | Vector | Overpass API via `osmnx` |
| **Places** | OpenStreetMap | Vector | Overpass API via `osmnx` |

> **Note**: JRC water bodies are used instead of OSM water features because JRC provides consistent, satellite-derived water extent data that captures seasonal water bodies and lagoons more reliably.

---

## Output Location

```
data/outputs/01_aoi_delineation/
├── Study_Area_Map_Publication.png/pdf
├── aoi_features_tracking.json
└── City_[Name]/
    └── Features/OSM/
        ├── [Name]_water_bodies.geojson   (JRC via GEE)
        ├── [Name]_roads.geojson          (OSM)
        └── [Name]_places.geojson         (OSM)
```

---

## Skip Logic

- **Area tracking**: Features only re-extracted if AOI area changes (tracked via `aoi_features_tracking.json`)
- **File checking**: Individual feature files verified before extraction

---

## Prerequisites

This notebook requires:
- **Notebook 01** completed first (AOI boundaries must exist)
- **Google Earth Engine** authentication for JRC water extraction
- **osmnx** package for OSM queries (`pip install osmnx`)

---
## 1. Load City AOIs

Load all city AOI boundaries from the `01_aoi_delineation` output folder and check which cities need feature updates based on tracking data.

In [ ]:
# ====================================================================
# IMPORT LIBRARIES AND CONFIGURE
# ====================================================================

import ee
import glob
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import sys
import time
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

# Import configuration and modules
import config
from src.gee_utils import initialize_gee
from src.data_io import (
    load_aoi_tracking, save_aoi_tracking, update_aoi_tracking,
    check_features_need_update, check_osm_feature_exists,
    load_city_osm_features, load_city_polygon_features,
    get_osm_extraction_status, get_osm_features_summary, 
    filter_places_for_display
)
from src.visualization import create_geospatial_features_map

print("="*80)
print("GEOSPATIAL FEATURES EXTRACTION")
print("="*80)
print(f"Project root: {project_root}")

# Initialize Google Earth Engine
service_key = config.GEE_CONFIG['service_json']
project_id = config.GEE_CONFIG['project_id']

GEE_AVAILABLE = initialize_gee(service_key, project_id)
if not GEE_AVAILABLE:
    print("  Water features will be skipped")

# Import osmnx for OSM queries via Overpass API
OSMNX_AVAILABLE = False
try:
    import osmnx as ox
    ox.settings.timeout = 180  # 3 min timeout
    ox.settings.memory = None
    ox.settings.log_console = False
    print("✓ osmnx available for Overpass API queries (10-30s per city)")
    OSMNX_AVAILABLE = True
except ImportError:
    print("✗ osmnx not available - install with: pip install osmnx")

print(f"GeoPandas version: {gpd.__version__}")
print("="*80)


In [ ]:
# ====================================================================
# LOAD AOI BOUNDARIES & CHECK TRACKING STATUS
# ====================================================================

aoi_base_dir = project_root / 'data' / 'outputs' / '01_aoi_delineation'

print(f"\nLoading AOI boundaries from: {aoi_base_dir}")
print("-"*80)

aoi_dict = {}
# AOI files are in City_*/Features/ subdirectory
aoi_pattern = str(aoi_base_dir / 'City_*' / 'Features' / '*_AOI_Boundary.geojson')
aoi_files = glob.glob(aoi_pattern)

if not aoi_files:
    raise FileNotFoundError(f"No AOI boundary files found in {aoi_base_dir}/City_*/Features/")

# Load current tracking data
tracking_data = load_aoi_tracking(aoi_base_dir)
print(f"\n📋 Tracking file: {len(tracking_data)} cities previously tracked")
if tracking_data:
    print(f"   Tracked cities: {', '.join(tracking_data.keys())}")

update_count = 0
skip_count = 0

for file in sorted(aoi_files):
    city_name = Path(file).stem.replace('_AOI_Boundary', '')
    gdf = gpd.read_file(file)
    
    # Ensure WGS84 for OSM queries
    if gdf.crs != 'EPSG:4326':
        gdf = gdf.to_crs('EPSG:4326')
    
    aoi_dict[city_name] = gdf
    bounds = gdf.total_bounds
    
    # Check if features need update (uses Main_Polygon area for comparison)
    needs_update, reason, current_area, tracked_area = check_features_need_update(city_name, aoi_base_dir)
    
    if needs_update:
        update_indicator = "🔄 UPDATE NEEDED"
        update_count += 1
    else:
        update_indicator = "✓ Up to date"
        skip_count += 1
    
    # Display Main_Polygon area (what's actually tracked)
    print(f"{update_indicator} {city_name:15s} | Main Polygon: {current_area:7.2f} km² | {reason}")

cities_to_process = sorted(aoi_dict.keys())

print("-"*80)
print(f"Loaded {len(aoi_dict)} city AOIs: {', '.join(cities_to_process)}")
if skip_count == len(cities_to_process):
    print(f"✓ All {skip_count} cities up-to-date - feature extraction will be SKIPPED")
elif update_count > 0:
    print(f"🔄 {update_count} cities need updates, {skip_count} will be skipped")
print("="*80)

In [ ]:
# ====================================================================
# PRE-EXTRACTION STATUS CHECK
# ====================================================================

print(f"\n{'='*80}")
print("FEATURE EXTRACTION STATUS CHECK")
print(f"{'='*80}")

for city in cities_to_process:
    status = get_osm_extraction_status(city, aoi_base_dir)
    
    print(f"\n{city}:")
    print(f"  AOI Area: {status['current_area_km2']:.2f} km²")
    print(f"  Needs Update: {status['needs_update']} ({status['update_reason']})")
    
    for feat_type, feat_info in status['features'].items():
        if feat_info['exists']:
            print(f"  ✓ {feat_type}: {feat_info['count']} features")
        else:
            print(f"  ✗ {feat_type}: Not extracted")

print(f"\n{'='*80}")
print("NOTE: Only features needing update will be extracted")
print(f"{'='*80}")

---
## 2. Extract Features for Each City

For each city AOI, extract three feature types:

| Feature | Source | Method | Threshold |
|---------|--------|--------|-----------|
| **Water Bodies** | JRC Global Surface Water | Google Earth Engine | ≥50% occurrence (captures lagoons) |
| **Roads** | OpenStreetMap | Overpass API via `osmnx` | All highway types |
| **Places** | OpenStreetMap | Overpass API via `osmnx` | All place types |

> **Skip Logic**: If a city's AOI area hasn't changed since the last extraction (tracked in `aoi_features_tracking.json`), that city will be skipped to save time.

In [ ]:
# ====================================================================
# EXTRACT FEATURES FOR EACH CITY (WITH SKIP LOGIC)
# ====================================================================

for i, city in enumerate(cities_to_process, 1):
    print(f"\n{'='*80}")
    print(f"PROCESSING {city.upper()} ({i}/{len(cities_to_process)})")
    print(f"{'='*80}")
    
    gdf = aoi_dict[city]
    
    # Check if this city needs any updates
    needs_update, reason, current_area, tracked_area = check_features_need_update(city, aoi_base_dir)
    
    # Create output directory
    city_osm_dir = aoi_base_dir / f'City_{city}' / 'Features' / 'OSM'
    city_osm_dir.mkdir(parents=True, exist_ok=True)
    
    # Get the AOI polygon for queries
    aoi_polygon = gdf.to_crs('EPSG:4326').union_all()
    
    # Track if any features were extracted
    features_extracted = False
    
    # ====================================================================
    # 1. JRC SURFACE WATER (FROM GEE)
    # ====================================================================
    water_exists, water_file, water_count = check_osm_feature_exists(city, 'water', aoi_base_dir)
    
    if water_exists and not needs_update:
        print(f"\n1. Water bodies: ⏭️  SKIP ({water_count} features exist, AOI unchanged)")
    elif not GEE_AVAILABLE:
        print(f"\n1. Water bodies: ✗ GEE not available")
    else:
        print(f"\n1. Downloading JRC surface water for {city}...")
        try:
            water_file_path = city_osm_dir / f'{city}_water_bodies.geojson'
            
            # Convert AOI to Earth Engine geometry
            aoi_geojson = gdf.to_crs('EPSG:4326').__geo_interface__
            ee_geometry = ee.Geometry(aoi_geojson['features'][0]['geometry'])
            
            # Load JRC Global Surface Water
            jrc_water = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
            water_occurrence = jrc_water.select('occurrence')
            water_mask = water_occurrence.gte(50)
            water_masked = water_mask.updateMask(water_mask.eq(1))
            
            # Convert to vectors
            water_vectors = water_masked.reduceToVectors(
                geometry=ee_geometry,
                scale=30,
                geometryType='polygon',
                eightConnected=False,
                maxPixels=1e9
            )
            
            water_features = water_vectors.getInfo()
            
            if water_features['features']:
                water_gdf = gpd.GeoDataFrame.from_features(water_features['features'], crs='EPSG:4326')
                water_clipped = gpd.clip(water_gdf, gdf.to_crs('EPSG:4326'))
                
                if not water_clipped.empty:
                    water_clipped['area_km2'] = water_clipped.to_crs('EPSG:32632').area / 1e6
                    water_clipped = water_clipped[water_clipped['area_km2'] >= 0.001]
                    
                    if not water_clipped.empty:
                        water_clipped.to_file(water_file_path, driver='GeoJSON')
                        total_area = water_clipped['area_km2'].sum()
                        print(f"   ✓ Saved {len(water_clipped)} water bodies ({total_area:.2f} km²)")
                        features_extracted = True
                    else:
                        print(f"   - No significant water bodies after filtering")
                else:
                    print(f"   - No water bodies within AOI")
            else:
                print(f"   - No water detected by JRC")
                
        except Exception as e:
            print(f"   ✗ Error: {e}")
    
    # ====================================================================
    # 2. OSM ROADS (VIA OVERPASS API)
    # ====================================================================
    roads_exists, roads_file, roads_count = check_osm_feature_exists(city, 'roads', aoi_base_dir)
    
    if roads_exists and not needs_update:
        print(f"\n2. Roads: ⏭️  SKIP ({roads_count} features exist, AOI unchanged)")
    elif not OSMNX_AVAILABLE:
        print(f"\n2. Roads: ✗ osmnx not available")
    else:
        print(f"\n2. Extracting OSM roads for {city}...")
        try:
            roads_file_path = city_osm_dir / f'{city}_roads.geojson'
            start_time = time.time()
            
            G = ox.graph_from_polygon(aoi_polygon, network_type='drive', simplify=True)
            roads = ox.graph_to_gdfs(G, nodes=False)
            elapsed = time.time() - start_time
            
            if not roads.empty:
                roads = roads.to_crs('EPSG:4326')
                roads.to_file(roads_file_path, driver='GeoJSON')
                print(f"   ✓ Saved {len(roads)} road segments in {elapsed:.1f}s")
                features_extracted = True
                
                if 'highway' in roads.columns:
                    highway_flat = roads['highway'].apply(
                        lambda x: x[0] if isinstance(x, (list, tuple)) else x
                    )
                    road_types = highway_flat.value_counts().head(5)
                    print(f"   Top types: {dict(road_types)}")
            else:
                print(f"   - No roads found")
                
        except Exception as e:
            elapsed = time.time() - start_time
            print(f"   ✗ Error after {elapsed:.1f}s: {e}")
    
    # ====================================================================
    # 3. OSM PLACES (VIA OVERPASS API)
    # ====================================================================
    places_exists, places_file, places_count = check_osm_feature_exists(city, 'places', aoi_base_dir)
    
    if places_exists and not needs_update:
        print(f"\n3. Places: ⏭️  SKIP ({places_count} features exist, AOI unchanged)")
    elif not OSMNX_AVAILABLE:
        print(f"\n3. Places: ✗ osmnx not available")
    else:
        print(f"\n3. Extracting OSM places for {city}...")
        try:
            places_file_path = city_osm_dir / f'{city}_places.geojson'
            start_time = time.time()
            
            place_tags = {'place': True}
            places = ox.features_from_polygon(aoi_polygon, tags=place_tags)
            elapsed = time.time() - start_time
            
            if not places.empty:
                places = places.to_crs('EPSG:4326')
                places.to_file(places_file_path, driver='GeoJSON')
                print(f"   ✓ Saved {len(places)} places in {elapsed:.1f}s")
                features_extracted = True
                
                if 'place' in places.columns:
                    place_types = places['place'].value_counts().head(5)
                    print(f"   Top types: {dict(place_types)}")
                
                if 'name' in places.columns:
                    named_places = places[places['name'].notna()]['name'].head(5)
                    if not named_places.empty:
                        print(f"   Sample names: {', '.join(named_places.tolist())}")
            else:
                print(f"   - No places found")
                
        except Exception as e:
            elapsed = time.time() - start_time
            if "EmptyOverpassResponse" in str(type(e).__name__) or "no data" in str(e).lower():
                print(f"   - No places found")
            else:
                print(f"   ✗ Error after {elapsed:.1f}s: {e}")
    
    # ====================================================================
    # UPDATE TRACKING IF FEATURES WERE EXTRACTED
    # ====================================================================
    if features_extracted or needs_update:
        update_aoi_tracking(city, current_area, aoi_base_dir)
        print(f"\n   📋 Tracking updated for {city} (area: {current_area:.2f} km²)")

print(f"\n{'='*80}")
print("FEATURE EXTRACTION COMPLETE")
print(f"{'='*80}")

---
## 3. Visualize Geospatial Features

Create maps showing extracted features for each city:

| Feature | Styling | Filter |
|---------|---------|--------|
| **Water** | Blue fill (`steelblue`) with dark edge | JRC ≥50% occurrence |
| **Roads** | Thin grey lines | All highway types |
| **Places** | Dark red markers with labels | Towns and suburbs only |

All visualization outputs are saved in both **PDF** and **PNG** formats to `City_[Name]/Figures/`.

In [ ]:
# ====================================================================
# SUMMARY OF DOWNLOADED FEATURES
# ====================================================================

print(f"\n{'='*80}")
print("GEOSPATIAL FEATURES SUMMARY")
print(f"{'='*80}")

# Get summary table using modular function
summary_df = get_osm_features_summary(cities_to_process, aoi_base_dir)

# Display formatted summary
for _, row in summary_df.iterrows():
    city = row['city']
    print(f"\n{city}:")
    print("-"*60)
    
    total_size = 0
    for feat in ['water', 'roads', 'places']:
        count = row[f'{feat}_count']
        size = row[f'{feat}_size_mb']
        total_size += size
        
        if count > 0:
            print(f"  ✓ {feat:10s}: {count:5d} features | {size:6.2f} MB")
        else:
            print(f"  ✗ {feat:10s}: Not available")
    
    print(f"  {'─'*50}")
    print(f"  Total: {total_size:.2f} MB")

# Store feature paths for visualization
geo_features = {}
for city in cities_to_process:
    geo_features[city] = load_city_osm_features(city, aoi_base_dir)

# Load polygon features for all cities
polygon_features = {}
for city in cities_to_process:
    polygon_features[city] = load_city_polygon_features(city, aoi_base_dir)

print(f"\n{'='*80}")
print(f"Output location: {aoi_base_dir}/City_[Name]/Features/OSM/")
print(f"Tracking file: {aoi_base_dir}/aoi_features_tracking.json")
print(f"{'='*80}")

In [ ]:
# ====================================================================
# VISUALIZE GEOSPATIAL FEATURES FOR EACH CITY
# ====================================================================

print(f"\n{'='*80}")
print("GEOSPATIAL FEATURES VISUALIZATION")
print(f"{'='*80}\n")

for city in cities_to_process:
    print(f"Visualizing {city}...")
    
    # Get AOI boundary
    aoi_gdf = aoi_dict[city]
    
    # Get features (already loaded)
    features = geo_features[city]
    polygons = polygon_features[city]
    
    # Filter places to towns and suburbs only
    places_filtered = filter_places_for_display(
        features.get('places'),
        place_types=['town', 'suburb']
    )
    
    # Create output path
    fig_dir = aoi_base_dir / f'City_{city}' / 'Figures'
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig_path = fig_dir / f'{city}_geospatial_features_map.png'
    
    # Create visualization using modular function with polygon layers
    # Use 2 sq km labelling (1 suburb per 2 sq km = ~1.4 km buffer radius)
    fig = create_geospatial_features_map(
        city_name=city,
        aoi_gdf=aoi_gdf,
        main_polygon_gdf=polygons.get('main_polygon'),
        adjacent_polygons_gdf=polygons.get('adjacent_polygons'),
        water_gdf=features.get('water'),
        roads_gdf=features.get('roads'),
        places_gdf=places_filtered,
        place_types_filter=['town', 'suburb'],
        label_buffer_km=1.4,  # ~2 sq km coverage per label (pi * 1.4^2 ≈ 6.2 sq km, but accounting for overlap)
        figsize=(12, 10),
        save_path=str(fig_path),
        dpi=300
    )
    
    plt.show()
    plt.close()

print(f"\n{'='*80}")
print("VISUALIZATION COMPLETE")
print(f"{'='*80}")
print("\n✓ All maps saved to PDF and PNG formats")
print("✓ JRC water bodies will be automatically masked during SEVIRI processing")
print("✓ OSM roads and places available for overlay visualization")

---
## 4. Publication-Ready Study Area Map

Create a comprehensive multi-panel study area map for manuscript publication:

| Element | Description |
|---------|-------------|
| **Nigeria inset** | Country outline with AOI locations and city labels |
| **City panels** | Dynamic grid arrangement with JRC water, OSM roads, and OSM places |
| **Layout** | A4 landscape format (11.69" × 7.5") |
| **Cartographic elements** | Scale bars inside map frames, north arrow, panel labels (A, B, C...) |

Output: `Study_Area_Map_Publication.png` and `Study_Area_Map_Publication.pdf`

In [ ]:
# ====================================================================
# PUBLICATION-READY STUDY AREA MAP (Modularized)
# ====================================================================

from src.visualization import create_publication_study_area_map

print(f"\n{'='*80}")
print("CREATING PUBLICATION-READY STUDY AREA MAP")
print(f"{'='*80}")

# Define paths
nigeria_path = project_root / 'data' / 'inputs' / 'geospatial_features' / 'boundary_Nigeria.geojson'
save_path = aoi_base_dir / 'Study_Area_Map_Publication'

# Create publication map using modular function
fig = create_publication_study_area_map(
    aoi_dict=aoi_dict,
    geo_features=geo_features,
    polygon_features=polygon_features,
    nigeria_path=nigeria_path,
    save_path=save_path,
    title='Study Area Maps: Nigerian Cities with Geospatial Features',
    figsize=(11.69, 7.5),  # A4 landscape
    dpi=300
)

plt.show()

print(f"\n{'='*80}")

---
## Summary

This notebook completed the **geospatial feature extraction and visualization** workflow for Nigerian city AOIs.

### Workflow Completed

| Step | Description | Output |
|------|-------------|--------|
| **1. Load AOIs** | Loaded city AOI boundaries from `01_aoi_delineation` | AOI GeoDataFrames with tracking status |
| **2. Feature Extraction** | Downloaded water (JRC/GEE), roads & places (OSM/Overpass) | GeoJSON files per city |
| **3. Individual Maps** | Created feature maps for each city | PNG + PDF per city |
| **4. Publication Map** | Multi-panel study area figure (A4 landscape) | `Study_Area_Map_Publication.png/pdf` |

### Data Sources Used

| Feature | Source | Why This Source? |
|---------|--------|------------------|
| **Water Bodies** | JRC Global Surface Water (GEE) | Consistent satellite-derived extent; captures seasonal water and lagoons reliably |
| **Roads** | OpenStreetMap (Overpass API) | Comprehensive road network; always up-to-date |
| **Places** | OpenStreetMap (Overpass API) | Rich place name database; includes towns, suburbs, villages |

### Output Structure

```
data/outputs/01_aoi_delineation/
├── Study_Area_Map_Publication.png    # Multi-panel publication figure
├── Study_Area_Map_Publication.pdf
├── aoi_features_tracking.json        # Skip logic tracking
└── City_[Name]/
    ├── Features/
    │   ├── [Name]_AOI_Boundary.geojson
    │   ├── [Name]_Main_Polygon.geojson
    │   ├── [Name]_Adjacent_Polygons.geojson
    │   └── OSM/
    │       ├── [Name]_water_bodies.geojson  ← JRC via GEE
    │       ├── [Name]_roads.geojson         ← OSM via Overpass
    │       └── [Name]_places.geojson        ← OSM via Overpass
    └── Figures/
        └── [Name]_geospatial_features_map.png/pdf
```

### Key Features

- **Skip Logic**: Features only re-extracted when AOI area changes (tracked via JSON)
- **Fast Queries**: Overpass API completes in 10-30s per city
- **Publication-Ready**: A4 landscape figure with scale bars inside map frames
- **Modularized**: Uses functions from `src/visualization.py` and `src/data_io.py`

### Next Steps

The extracted geospatial features will be used in subsequent notebooks:
- **Water bodies** → Mask during LST processing (avoid water pixel contamination)
- **Roads/Places** → Overlay on LST hotspot maps for context
- **Publication map** → Include in manuscript methods section

In [ ]:
# ====================================================================
# DEBUG: CHECK POLYGON ATTRIBUTES FOR EACH CITY
# ====================================================================
# Run this cell only when you need to inspect polygon attributes

print("="*80)
print("POLYGON ATTRIBUTES CHECK")
print("="*80)

for city in cities_to_process:
    print(f"\n{'─'*60}")
    print(f"📍 {city}")
    print(f"{'─'*60}")
    
    polygons = polygon_features[city]
    
    # Check main_polygon
    main_poly = polygons.get('main_polygon')
    if main_poly is not None and not main_poly.empty:
        print(f"\n  MAIN POLYGON:")
        print(f"    Rows: {len(main_poly)}")
        print(f"    Columns: {list(main_poly.columns)}")
        print(f"    CRS: {main_poly.crs}")
        print(f"    Geometry type: {main_poly.geometry.geom_type.unique().tolist()}")
        # Show all attributes (excluding geometry)
        non_geom_cols = [c for c in main_poly.columns if c != 'geometry']
        if non_geom_cols:
            print(f"    Attributes (first row):")
            for col in non_geom_cols:
                val = main_poly[col].iloc[0] if len(main_poly) > 0 else None
                print(f"      • {col}: {val}")
    else:
        print(f"\n  MAIN POLYGON: None or empty")
    
    # Check adjacent_polygons
    adj_poly = polygons.get('adjacent_polygons')
    if adj_poly is not None and not adj_poly.empty:
        print(f"\n  ADJACENT POLYGONS:")
        print(f"    Rows: {len(adj_poly)}")
        print(f"    Columns: {list(adj_poly.columns)}")
        print(f"    CRS: {adj_poly.crs}")
        print(f"    Geometry type: {adj_poly.geometry.geom_type.unique().tolist()}")
        # Show all attributes (excluding geometry)
        non_geom_cols = [c for c in adj_poly.columns if c != 'geometry']
        if non_geom_cols:
            print(f"    Attributes (first row):")
            for col in non_geom_cols:
                val = adj_poly[col].iloc[0] if len(adj_poly) > 0 else None
                print(f"      • {col}: {val}")
            # If multiple rows, show count of unique values per column
            if len(adj_poly) > 1:
                print(f"    All adjacent polygon names:")
                if 'Urban Agglomeration' in adj_poly.columns:
                    for i, name in enumerate(adj_poly['Urban Agglomeration'].tolist()):
                        print(f"      {i+1}. {name}")
    else:
        print(f"\n  ADJACENT POLYGONS: None or empty")

print(f"\n{'='*80}")